In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import librosa
import itertools
import io
import numpy as np
import json

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [7]:
files = glob('ArVoice/data/*')
len(files)

39

In [14]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'

    files, _ = files

    data = []
    for f in tqdm(files):
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in range(len(df)):
            try:
                t = df['transcription'].iloc[i].strip()
                if len(t) < 2:
                    continue
                audio_filename = f'{f_new}_{i}.mp3'
                audio_filename = os.path.join(base, audio_filename)
                b = df['normalized_wav'].iloc[i]['bytes']
                audio_np, sr = sf.read(io.BytesIO(b))
                if audio_np.ndim > 1:
                    audio_np = audio_np.mean(axis=1)
                if audio_np.shape[0] < 10000:
                    continue
                sf.write(audio_filename, audio_np, sr)
                
                data.append({
                    'audio_filename': audio_filename,
                    'text': df['transcription'].iloc[i],
                    'speaker': f"{base}_{df['speaker_id'].iloc[i]}"
                })
            except Exception as e:
                pass
        
    return data

In [15]:
data = loop((files[:1], 0))

100%|██████████| 1/1 [00:34<00:00, 34.56s/it]


In [16]:
data = multiprocessing(files, loop, cores = len(files))

100%|██████████| 1/1 [00:45<00:00, 45.17s/it]


In [17]:
len(data)

23111

In [18]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'audio_filename': 'ArVoice_audio/ArVoice-data-train-00022-of-00035_0.mp3',
 'text': 'وَرِعَايَةٍ صِحِّيَّةٍ تُحَقِّقُ حَيَاةً أَوْفَرَ حَظًّا في الْقَضَاءِ عَلَى الْأَمْرَاضِ الْمُزْمِنَةِ',
 'speaker': 'ArVoice_audio_ar-XA-Wavenet-B'}

In [19]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'ArVoice')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 96.18ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████|  591kB /  591kB, 2.95MB/s  
Processing Files (1 / 1): 100%|██████████|  591kB /  591kB, 1.48MB/s  
New Data Upload: 100%|██████████|  591kB /  591kB, 1.48MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.26 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/fe05f1241652d769120c5d0cbf600a4ab315a197', commit_message='Upload dataset', commit_description='', oid='fe05f1241652d769120c5d0cbf600a4ab315a197', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [20]:
audio_files = [d['audio_filename'] for d in data]

with open('ArVoice-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [22]:
folders = glob('ArVoice_audio*')
folders = [f for f in folders if '.zip' not in f]
for f in folders:
    print(f)
    os.system(f'zip -rq {f}.zip {f}')

ArVoice_audio_neucodec
ArVoice_audio


In [23]:
from huggingface_hub import HfApi
api = HfApi()

for f in glob('ArVoice_audio*.zip'):
    api.upload_file(
        path_or_fileobj=f,
        path_in_repo=f,
        repo_id="malaysia-ai/Multilingual-TTS",
        repo_type="dataset",
    )

Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):   1%|          | 13.1MB / 1.17GB,   ???B/s  
Processing Files (0 / 1):  12%|█▏        |  135MB / 1.17GB,  610MB/s  
Processing Files (0 / 1):  13%|█▎        |  150MB / 1.17GB,  342MB/s  
Processing Files (0 / 1):  22%|██▏       |  258MB / 1.17GB,  408MB/s  
Processing Files (0 / 1):  34%|███▎      |  394MB / 1.17GB,  477MB/s  
Processing Files (0 / 1):  44%|████▎     |  510MB / 1.17GB,  497MB/s  
Processing Files (0 / 1):  44%|████▍     |  512MB / 1.17GB,  416MB/s  
Processing Files (0 / 1):  44%|████▍     |  517MB / 1.17GB,  360MB/s  
Processing Files (0 / 1):  44%|████▍     |  519MB / 1.17GB,  316MB/s  
Processing Files (0 / 1):  45%|████▍     |  523MB / 1.17GB,  283MB/s  
Processing Files (0 / 1):  45%|████▍     |  526MB / 1.17GB,  256MB/s  
Processing Files (0 / 1):  45%|████▌     |  529MB / 1.17GB,  235MB/s  
Processing Files (0 / 1):  45%|████▌     |  531MB / 1.17GB,  216MB/s  
Processing